# Primera lectura CSV y limpieza de los mismos

Lectura CSV préstamos. Inicialmente, encuentran diversos fallos en la deteccion de decodificacion, se resuelen. Se concatenan los archivos para generar un unico DF, en la misma lectura se filtra por libros

Errores observados en lecturas CSV:

- Nombre columnas incorrectos: en vez de comenzar por prXXX comienzan por phXXX. Se sustituye ph por pr si coincidencia:
        (- PRESTAMOS/2025_05_mayo.csv
        - PRESTAMOS/2025_07_julio.csv
        - PRESTAMOS/2025_06_junio.csv
        - PRESTAMOS/2025_08_agosto.csv
        - PRESTAMOS/2025_09_septiembre.csv)
- Decodificacion:
- Separadores:
    - Existen ; en campo titulo sin comillas, se detecta como separacion y se pasa a nueva columna.
    - Campos, con ; que aun estar entrecomillados fallan. Por detección quoting quote_none o valor quotchear distinto al registrado en función.
    - Filas con un campo vacio de más, en distintas posiciones.

Previamente a la limpieza se revisan comillas en texto y doble codificacion.

En el proceso de limpieza se establece formato datos category en los campos de códigos de informacion: El DF final pesa mucho con mas de 18 millones de registros y 14 campos. Se procede a pasar como tipo de dato category los campos : prcocp, prprsu, prcolp,	prlebi,	prlesu,	pradul,	prcocs, prcoc1 y archivo_origen.

Posteriormente a la primera limpieza se realizará la limpieza independiente de la tabla generada. Una vez terminada, se guarda DF en formato parquet, en carpeta silver.

El tratamiento independiente de la tabla, similar a los observados en el resto de tablas:
1. Duplicados completos
2. Duplicados parciales
3. Comprobación valores nulos junto con asignación de valor real o nulo explícito
4. Asignación de los tipos de datos correctos.
    - Si pocos valores únicos repetidos en diversas filas -> category
    - Datos texto -> formato string pyarrow, para reducir memoria y velocidad en operaciones str.
    - Fechas -> Datetime
    - Solo años -> Int64
5. Normalización máxima de los campos de título, autor e ISBN en tablas registros y préstamos, eliminando signos y símbolos de puntuación
6. Eliminación de las columnas no necesarias


# ACCIONES INICIALES

In [ ]:
!pip install -q ftfy

In [ ]:
from google.colab import drive # montar drive
import shutil # copiar docs. drive en disco local
from pathlib import Path # Gestionar rutas archivos
import csv # Ver formato csv
import json # Para crear y tratar archivo json
import pandas as pd
#modulo estandar de python. Counter (calse) cuenta cuantas veces aparece un
# elemento en un iterable:
from collections import Counter
from collections.abc import Iterable # Para poder iterar las rutas
import re # modulo de expresiones regulares
import unicodedata # Para normalizar strings
import numpy as np
import gc
from functools import partial
import ftfy
from IPython.display import display

pd.options.mode.copy_on_write = True
pd.set_option("mode.string_storage", "pyarrow")
pd.set_option("io.parquet.engine", "pyarrow")



In [ ]:
# Montar drive:
drive.mount('/content/drive')

## Constantes


In [ ]:
# 1. Rutas originales
pf = Path('drive')
local = Path('/content')

# 2. Rutas sobre las que trabajar:
# 2.1 Rutas que apuntan hacia Drive
DRIVE_ORIGEN = pf / 'bronze'
DRIVE_DESTINO = pf / 'silver'

# 2.2 Rutas que apuntan hacia disco local:
ORIGEN_LOCAL = local / 'datos_bronze'
DESTINO_LOCAL = local / 'resultados_silver'
ORIGEN_LOCAL.mkdir(parents=True, exist_ok=True)
DESTINO_LOCAL.mkdir(parents=True, exist_ok=True)


# 3. Lectura de los CSV de préstamos:

# 3.1 parametros read_csv extra a tener en cuenta al leerlos.
PARAMS_BRONZE = {'dtype': 'string', 'keep_default_na': False}

CHUNKSIZE = 200_000                       # lectura csv por bloque
DIR_PARTES = local / "partes_prestamos"   # parquet intermedios en disco local
DIR_SANEADOS = local / "bronze_saneado"

COL_SOPORTE = "prcocs"
SOPORTE_OBJETIVO = "LIB"
COL_TITULO = "tititu"
COL_AUTOR = "tiauto"
COL_ISBN = "tiisxn"
COL_PUBLICO = "prcoc1"
COLUMNAS_TEXTO = (COL_TITULO, COL_AUTOR)   # campos libres que pueden contener ";"
PREFIJO_ANTIGUO = "ph"
PREFIJO_ACTUAL = "pr"
RENOMBRAR_ERRATAS = {"rpcocp": "prcocp"}   # erratas de cabecera conocidas
QUOTECHAR_PRESTAMOS = '"'

# Entidad HTML numérica partida por su ";" final: "Ste&#769" | "phanie"
PATRON_ENTIDAD = re.compile(r"&#x?[0-9A-Fa-f]+$")
# ISBN válido en bronze: dígitos/X/guiones, opcionalmente seguido de un
# calificador truncado que empieza por "(" o ":" ("978-84-... (", "84-... :")
PATRON_ISBN = r"\s*[0-9Xx\-]*\s*(?:[(:].*)?"
# ISBN convertido a notación científica por Excel ("9,78841E+12")
PATRON_ISBN_CIENTIFICO = r"\d,\d*E\+\d+"

COLUMNAS_RESUMEN = [
    "antes", "despues", "descartadas",
    "lineas_reparadas", "lineas_desc_objetivo", "lineas_desc_otras",
    "filas_cortas", "isbn_cientifico", "isbn_raros", "cambios",
]


# 4. Limpieza y transformaciones DF
PATRONES_NULOS = ("", "-", ".", "?", "n/a", "na", "n.a", "n.a.",
                  "null", "none", "nan","sd", "s/d", "sin datos",
                  "tbd", "x", "xx", "xxx")



## Copia archivos drive en local

In [ ]:
# Copia de todas las carpetas de drive origen o solo las seleccionadas
COPIAR = ["PRESTAMOS_HISTORICOS"]

assert DRIVE_ORIGEN.exists(), f"No existe {DRIVE_ORIGEN}"

disponibles = sorted(p.name for p in DRIVE_ORIGEN.iterdir() if p.is_dir())
seleccion = COPIAR or disponibles

faltan = [s for s in seleccion if s not in disponibles]
assert not faltan, f"No existen en DRIVE_ORIGEN: {faltan}. Disponibles: {disponibles}"

for sub in seleccion:
    shutil.copytree(DRIVE_ORIGEN / sub, ORIGEN_LOCAL / sub, dirs_exist_ok=True)

print(f"Subcarpetas en Drive: {disponibles}")
print(f"Copiadas: {seleccion}\n")

for sub in seleccion:
    csvs = sorted(p.name for p in (ORIGEN_LOCAL / sub).rglob("*.csv"))
    print(f"{sub} ({len(csvs)} csv): {csvs}")

In [ ]:
# Importacion archivo JSON desde Drive:

jsons = list(DRIVE_ORIGEN.glob('*.json'))
assert jsons, f'No hay ningún .json en {DRIVE_ORIGEN}'
for f in jsons:
    shutil.copy2(f, ORIGEN_LOCAL / f.name)
print(f"JSON copiados: {[f.name for f in jsons]}")


# Lectura archivo JSON para emplarlo en lectura CSV:

with open(ORIGEN_LOCAL / 'parametros_csv_brutos.json', encoding='utf-8') as f:
    params_json = json.load(f)

# Funciones


### Comprobacion carpetas y csv guardados en local

In [ ]:
# COMPROBAR CANTIDAD CSVs EN CARPETAS, SUBCARPETAS Y REGISTROS EN ARCHIVO JSON
def mostrar_resumen(df: pd.DataFrame, carpetas: list[str]) -> None:
    """
    Imprime la información obtenida en la función resumen_carpetas_inicio.
    """
    print(f"Se han importado las siguientes carpetas: \n{carpetas}\n")
    print("\nTabla resumen de CSVs encontrados en cada carpeta y coincidencia "
    "en las claves del diccionario JSON, así como sus diferencias:\n")
    display(df)


def resumen_carpetas_inicio(ruta: Path,
                            params: dict) -> list[str]:
    """
    Resume la cantidad de CSV que hay por carpeta y la coincidencia de las
    extensiones en el archivo JSON, mostrando diferencias de haberlas.
    Retorna lista de nombres carpetas presentes y muestra el resumen de CSVs
    """
    csvs = [
        f for f in ruta.rglob('*') if f.is_file() and f.suffix.lower() == '.csv'
        ]
    en_drive = Counter(f.parent.relative_to(ruta).as_posix() for f in csvs)

    # JSON: carpeta de cada clave registrada
    en_json = Counter(Path(r).parent.as_posix() for r in params)

    cantidad_csv = (pd.DataFrame({'en_drive': en_drive, 'en_json': en_json})
                  .fillna(0).astype(int)
                  .assign(diferencia=lambda d: d['en_drive'] - d['en_json'])
                  .reset_index(names=["Sub_carpeta"]))
    carpetas = cantidad_csv.loc[cantidad_csv["en_drive"] !=0, 'Sub_carpeta' ].tolist()
    mostrar_resumen(cantidad_csv, carpetas)
    return carpetas

### Lectura de CSVs

In [ ]:
def comprobar_nombre_carpetas(carpeta: str,
                              carpetas_disponibles: list[str]) -> list[str]:
    """Valida que `carpeta` sea una de las carpetas disponibles.

    Returns
    -------
    list of str
        Lista con la carpeta validada.

    Raises
    ------
    TypeError
        Si `carpeta` no es str.
    ValueError
        Si `carpeta` está vacía o no figura en `carpetas_disponibles`.
    """
    if not isinstance(carpeta, str):
        raise TypeError(
            f"'carpeta' debe ser str; se recibió {type(carpeta).__name__}"
        )
    if not carpeta.strip():
        raise ValueError("No se ha indicado ninguna carpeta.")
    if carpeta not in carpetas_disponibles:
        raise ValueError(
            f"Carpeta no disponible: {carpeta!r}. "
            f"Opciones válidas: {carpetas_disponibles}"
        )
    return [carpeta]


# ---------------------------------------------------------------------------
# Reparación de líneas con campos de más
# ---------------------------------------------------------------------------
def _indice_soporte(columnas: list[str]) -> int | None:
    """Posición de la columna de soporte (con prefijo antiguo o actual)."""
    nombres = {
        COL_SOPORTE,
        PREFIJO_ANTIGUO + COL_SOPORTE[len(PREFIJO_ACTUAL):],
    }
    return next((i for i, c in enumerate(columnas) if c in nombres), None)


def reparar_fila(campos: list[str],
                 columnas: list[str],
                 registro: list[tuple[str, list[str]]]) -> list[str] | None:
    """Repara una línea con más campos que columnas.

    Se usa como `on_bad_lines` de `pd.read_csv(engine="python")` y
    también para las filas desplazadas a columnas "Unnamed".
    Aplica, en este orden y acumulando, tres correcciones:

    D. Entidades HTML numéricas partidas por su ";" ("Ste&#769" |
       "phanie"): se vuelven a unir con ";".
    A. ";" sin comillas dentro de un campo de texto libre
       (COLUMNAS_TEXTO): se unen con ";" los trozos siguientes que
       empiezan por espacio (separador " ; "), hasta cubrir los campos
       de más.
    C. Si aún sobran campos, se eliminan los primeros campos vacíos
       situados después del primer campo de texto.

    Si al final el número de campos coincide con el de columnas, devuelve
    la línea reparada; si no, la descarta (devuelve None).

    Cada línea tratada se guarda en `registro` como (tipo, campos
    originales), con tipo:
    - "reparada:<pasos>" (p. ej. "reparada:entidad+texto_tititu")
    - "descartada_objetivo": no reparada y con soporte SOPORTE_OBJETIVO
    - "descartada_otro": no reparada y con otro soporte
    """
    if len(campos) <= len(columnas):
        return campos

    original = list(campos)
    campos = list(campos)
    pasos = []

    # D. Entidades HTML partidas
    j = 0
    while len(campos) > len(columnas) and j < len(campos) - 1:
        if PATRON_ENTIDAD.search(campos[j]):
            campos[j:j + 2] = [campos[j] + ";" + campos[j + 1]]
            pasos.append("entidad")
        else:
            j += 1

    # A. " ; " dentro de un campo de texto
    for col in COLUMNAS_TEXTO:
        extra = len(campos) - len(columnas)
        if extra <= 0:
            break
        if col not in columnas:
            continue
        i = columnas.index(col)
        k = 0
        while (k < extra and i + 1 + k < len(campos)
               and campos[i + 1 + k].startswith(" ")):
            k += 1
        if k:
            campos[i:i + k + 1] = [";".join(campos[i:i + k + 1])]
            pasos.append(f"texto_{col}")

    # C. Campos vacíos sobrantes
    extra = len(campos) - len(columnas)
    if extra > 0:
        inicio = min(
            (columnas.index(c) for c in COLUMNAS_TEXTO if c in columnas),
            default=0,
        )
        vacios = [j for j in range(inicio + 1, len(campos))
                  if campos[j] == ""]
        if len(vacios) >= extra:
            quitar = set(vacios[:extra])
            campos = [c for j, c in enumerate(campos) if j not in quitar]
            pasos.append("vacio")

    if len(campos) == len(columnas):
        registro.append(("reparada:" + "+".join(pasos), original))
        return campos

    i_sop = _indice_soporte(columnas)
    soporte = original[i_sop].strip() if i_sop is not None else None
    tipo = ("descartada_objetivo" if soporte == SOPORTE_OBJETIVO
            else "descartada_otro")
    registro.append((tipo, original))
    return None

# ---------------------------------------------------------------------------
# Armonización de la estructura
# ---------------------------------------------------------------------------
def armonizar_columnas(
        df: pd.DataFrame,
        registro: list[tuple[str, list[str]]]) -> tuple[pd.DataFrame,
                                                         list[str]]:
    """Unifica la estructura de columnas entre meses.

    1. Renombra el prefijo antiguo "ph" a "pr" (phcocs → prcocs...).
    2. Corrige erratas conocidas de cabecera (RENOMBRAR_ERRATAS).
    3. Elimina columnas repetidas en la cabecera (pandas las renombra
       "col.1") si están vacías. Si tienen datos, lanza error.
    4. Columnas "Unnamed" (cabecera terminada en ";"): las filas con
       datos en ellas son filas desplazadas por un ";" de más; se
       reparan con `reparar_fila` (o se descartan y registran) y
       después se eliminan las columnas "Unnamed".
    5. Si no existe `COL_PUBLICO` y `COL_ISBN` contiene tabuladores,
       separa "isbn\\tpublico" en dos columnas.

    Returns
    -------
    df : pd.DataFrame
    cambios : list of str
        Descripción de los cambios aplicados.

    Raises
    ------
    ValueError
        Si el renombrado genera columnas duplicadas, si una columna
        repetida tiene datos o si las columnas "Unnamed" no están al final.
    """
    cambios = []

    # 1. Prefijo ph -> pr
    if any(c.startswith(PREFIJO_ANTIGUO) for c in df.columns):
        df = df.rename(columns=lambda c: (
            PREFIJO_ACTUAL + c[len(PREFIJO_ANTIGUO):]
            if c.startswith(PREFIJO_ANTIGUO) else c
        ))
        cambios.append(f"prefijo {PREFIJO_ANTIGUO}→{PREFIJO_ACTUAL}")

    # 2. Erratas de cabecera
    erratas = {k: v for k, v in RENOMBRAR_ERRATAS.items() if k in df.columns}
    if erratas:
        df = df.rename(columns=erratas)
        cambios.append(f"erratas corregidas {erratas}")

    if df.columns.duplicated().any():
        raise ValueError(
            "Columnas duplicadas tras armonizar: "
            f"{df.columns[df.columns.duplicated()].tolist()}"
        )

    # 3. Columnas repetidas en la cabecera ("prfpre.1")
    repetidas = [
        c for c in df.columns
        if re.fullmatch(r".+\.\d+", str(c))
        and str(c).rsplit(".", 1)[0] in df.columns
    ]
    if repetidas:
        con_datos = [c for c in repetidas
                     if df[c].fillna("").ne("").any()]
        if con_datos:
            raise ValueError(f"Columnas repetidas con datos: {con_datos}")
        df = df.drop(columns=repetidas)
        cambios.append(f"eliminadas columnas repetidas vacías {repetidas}")

    # 4. Columnas sin nombre y filas desplazadas
    sin_nombre = [c for c in df.columns if str(c).startswith("Unnamed")]
    if sin_nombre:
        validas = [c for c in df.columns if c not in sin_nombre]
        if df.columns.tolist() != validas + sin_nombre:
            raise ValueError(
                f"Columnas sin nombre fuera del final: {sin_nombre}"
            )

        con_datos = df[sin_nombre].fillna("").ne("").any(axis=1)
        if con_datos.any():
            filas_descartadas = []
            n_reparadas = 0
            for idx in df.index[con_datos]:
                campos = df.loc[idx].fillna("").tolist()
                while len(campos) > len(validas) and campos[-1] == "":
                    campos.pop()
                reparada = reparar_fila(campos, validas, registro)
                if reparada is None:
                    filas_descartadas.append(idx)
                else:
                    df.loc[idx, validas] = reparada
                    n_reparadas += 1
            df = df.drop(index=filas_descartadas)
            cambios.append(
                f"filas desplazadas: {n_reparadas} reparadas, "
                f"{len(filas_descartadas)} descartadas"
            )
        df = df.drop(columns=sin_nombre)
        cambios.append(f"eliminadas columnas {sin_nombre}")

    # 5. Público unido al ISBN con tabulador
    if COL_PUBLICO not in df.columns and COL_ISBN in df.columns:
        tiene_tab = (df[COL_ISBN]
                     .str.contains("\t", regex=False)
                     .fillna(False))
        if tiene_tab.any():
            partes = df[COL_ISBN].str.split("\t", n=1, expand=True)
            df = df.assign(**{COL_ISBN: partes[0], COL_PUBLICO: partes[1]})
            cambios.append(
                f"{COL_PUBLICO} separado de {COL_ISBN} "
                f"({int(tiene_tab.sum())} de {len(df)} filas)"
            )

    return df, cambios


# ---------------------------------------------------------------------------
# Lectura de todos los CSV
# ---------------------------------------------------------------------------


def _procesar_trozos(lector, nombre: str,
                     registro: list, destino: Path) -> tuple[Counter, list[str]]:
    """Armoniza, filtra y vuelca a parquet cada trozo del lector."""
    stats = Counter()
    cambios = {}                          # dict como conjunto ordenado
    for i, chunk in enumerate(lector):
        stats["filas_cortas"] += int(chunk.isna().any(axis=1).sum())
        chunk, cambios_cols = armonizar_columnas(chunk, registro)
        cambios.update(dict.fromkeys(cambios_cols))
        stats["antes"] += len(chunk)

        filtrado = chunk[chunk[COL_SOPORTE].str.strip() == SOPORTE_OBJETIVO]
        del chunk

        if COL_ISBN in filtrado.columns:
            isbn = filtrado[COL_ISBN]
            cientifico = isbn.str.contains(PATRON_ISBN_CIENTIFICO).fillna(False)
            formato_ok = isbn.str.fullmatch(PATRON_ISBN).fillna(True)
            stats["isbn_cientifico"] += int(cientifico.sum())
            stats["isbn_raros"] += int((~formato_ok & ~cientifico).sum())
        else:
            stats["sin_isbn"] = 1

        stats["despues"] += len(filtrado)
        if len(filtrado):
            filtrado.to_parquet(destino / f"{nombre}_{i:04d}.parquet",
                                index=False)
        del filtrado
    return stats, list(cambios)

def procesar_archivo(ruta: Path, kwargs: dict, destino: Path):
    """Motor C por trozos → motor python con reparación → python sin comillas."""
    nombre = ruta.stem
    registro: list[tuple[str, list[str]]] = []

    def reiniciar():
        for p in destino.glob(f"{nombre}_*.parquet"):
            p.unlink()
        registro.clear()

    def con_python(kw: dict):
        columnas = pd.read_csv(ruta, nrows=0, **kw).columns.tolist()
        reparador = partial(reparar_fila, columnas=columnas, registro=registro)
        lector = pd.read_csv(ruta, engine="python", on_bad_lines=reparador,
                             chunksize=CHUNKSIZE, **kw)
        return _procesar_trozos(lector, nombre, registro, destino)

    try:
        lector = pd.read_csv(ruta, chunksize=CHUNKSIZE, **kwargs)
        stats, cambios = _procesar_trozos(lector, nombre, registro, destino)
    except pd.errors.ParserError:
        reiniciar()
        try:
            stats, cambios = con_python(kwargs)
        except csv.Error:
            reiniciar()
            stats, cambios = con_python({**kwargs, "quoting": csv.QUOTE_NONE})
            cambios = ["quoting=QUOTE_NONE (comillas literales)"] + cambios
    return stats, cambios, registro

def leer_csv(params: dict[str, dict], rutas: Iterable[Path], raiz: Path,
             destino: Path = DIR_PARTES):
    destino.mkdir(parents=True, exist_ok=True)
    for p in destino.glob("*.parquet"):   # limpia partes de ejecuciones previas
        p.unlink()

    d_fallos: dict[str, str] = {}
    d_resumen: dict[str, dict] = {}
    d_lineas: dict[str, list] = {}

    for ruta in rutas:
        final_ruta = ruta.relative_to(raiz).as_posix()
        nombre_archivo = ruta.stem
        info_params = params.get(final_ruta)

        if info_params is None:
            d_fallos[final_ruta] = "sin parámetros en el JSON"; continue
        if info_params.get("error"):
            d_fallos[final_ruta] = info_params["error"]; continue
        if "read_kwargs" not in info_params:
            d_fallos[final_ruta] = "sin 'read_kwargs' en el JSON"; continue
        if nombre_archivo in d_resumen:
            d_fallos[final_ruta] = f"archivo {nombre_archivo} ya leído"; continue

        read_kwargs = info_params["read_kwargs"]
        kwargs = {**read_kwargs, "quotechar": QUOTECHAR_PRESTAMOS,
                  **PARAMS_BRONZE}
        cambios = []
        quote_json = read_kwargs.get("quotechar", QUOTECHAR_PRESTAMOS)
        if quote_json != QUOTECHAR_PRESTAMOS:
            cambios.append(f"quotechar {quote_json}→{QUOTECHAR_PRESTAMOS}")

        try:
            ruta_lectura, lineas_saneadas = sanear_comillas(ruta)
            if lineas_saneadas:
                cambios.append(
                    f"comillas quitadas en líneas {lineas_saneadas[:10]}"
                    f" ({len(lineas_saneadas)} en total)"
                )
            stats, cambios_cols, lineas = procesar_archivo(ruta_lectura,
                                                           kwargs, destino)
        except Exception as e:
            for p in destino.glob(f"{nombre_archivo}_*.parquet"):
                p.unlink()
            d_fallos[final_ruta] = repr(e)
            continue
        finally:
            gc.collect()

        tipos = Counter(tipo.split(":")[0] for tipo, _ in lineas)
        sin_isbn = bool(stats["sin_isbn"])
        d_resumen[nombre_archivo] = {
            "antes": stats["antes"],
            "despues": stats["despues"],
            "lineas_reparadas": tipos["reparada"],
            "lineas_desc_objetivo": tipos["descartada_objetivo"],
            "lineas_desc_otras": tipos["descartada_otro"],
            "filas_cortas": stats["filas_cortas"],
            "isbn_cientifico": None if sin_isbn else stats["isbn_cientifico"],
            "isbn_raros": None if sin_isbn else stats["isbn_raros"],
            "cambios": "; ".join(cambios + cambios_cols),
        }
        if lineas:
            d_lineas[nombre_archivo] = lineas

    partes = sorted(destino.glob("*.parquet"))
    if not partes:
        return pd.DataFrame(), d_fallos, pd.DataFrame(columns=COLUMNAS_RESUMEN), d_lineas

    df_prestamos = pd.concat((pd.read_parquet(p) for p in partes),
                             ignore_index=True)

    resumen = pd.DataFrame.from_dict(d_resumen, orient="index")
    resumen["descartadas"] = resumen["antes"] - resumen["despues"]
    resumen = resumen[COLUMNAS_RESUMEN]
    resumen.loc["total"] = resumen.select_dtypes("number").sum()
    numericas = [c for c in COLUMNAS_RESUMEN if c != "cambios"]
    resumen[numericas] = resumen[numericas].astype("Int64")
    return df_prestamos, d_fallos, resumen, d_lineas
# ---------------------------------------------------------------------------
# Resumen y orquestador
# ---------------------------------------------------------------------------
def resumen_valores_lectura(df: pd.DataFrame,
                            d_fallos: dict[str, str],
                            resumen: pd.DataFrame,
                            d_lineas: dict[str, list]) -> None:
    """Muestra el resultado de la lectura, las correcciones y los fallos."""
    if resumen.empty:
        print("No se ha leído ningún CSV de préstamos.")
    else:
        print(f"CSV de préstamos leídos: {len(resumen) - 1}")
        print(f"DF final: {df.shape[0]} filas y {df.shape[1]} columnas")
        print("Filas descartadas por el filtro: "
              f"{resumen.loc['total', 'descartadas']}\n")
        display(resumen)
        if d_lineas:
            print("\nArchivos con líneas reparadas o descartadas: "
                  f"{list(d_lineas)}")

    if d_fallos:
        print(f"\nCSV NO CONVERTIDOS. Totales: {len(d_fallos)}")
        for ruta, motivo in d_fallos.items():
            print(f"{ruta}: {motivo}")


def lectura_prestamos(params: dict[str, dict],
                      carpeta: str,
                      carpetas_disponibles: list[str],
                      raiz: Path = ORIGEN_LOCAL,
                      mostrar: bool = True) -> tuple[pd.DataFrame,
                                                     dict[str, str]]:
    """Lee y corrige todos los CSV de préstamos de `carpeta` y subcarpetas.

    Ver `leer_csv` para el detalle de cada paso.

    Returns
    -------
    df_prestamos, d_fallos
    """
    carpetas = comprobar_nombre_carpetas(carpeta, carpetas_disponibles)
    rutas = sorted(
        r for c in carpetas for r in (raiz / c).rglob("*")
        if r.is_file() and r.suffix.lower() == ".csv"
    )
    df_prestamos, d_fallos, resumen, d_lineas = leer_csv(params, rutas, raiz)
    if mostrar:
        resumen_valores_lectura(df_prestamos, d_fallos, resumen, d_lineas)

    return df_prestamos, d_fallos

In [ ]:
def sanear_comillas(ruta: Path, destino: Path = DIR_SANEADOS) -> tuple[Path, list[int]]:
    """Copia el CSV quitando las comillas de las líneas con número impar de '"'.

    Una comilla impar en una línea es un entrecomillado truncado o mal
    exportado. Si no se corrige, el motor C fusiona líneas hasta la
    siguiente comilla. Si no hay líneas impares, devuelve la ruta original.
    """
    with open(ruta, "rb") as f:
        impares = [n for n, linea in enumerate(f, 1) if linea.count(b'"') % 2]
    if not impares:
        return ruta, []

    destino.mkdir(parents=True, exist_ok=True)
    salida = destino / ruta.name
    marcadas = set(impares)
    with open(ruta, "rb") as fin, open(salida, "wb") as fout:
        for n, linea in enumerate(fin, 1):
            fout.write(linea.replace(b'"', b"") if n in marcadas else linea)
    return salida, impares

### Limpieza DF prestamos

In [ ]:
# Transformaciones iniciales PRESTAMOS_HISTORICOS

# 1. MOSTRAR DF  INFORMACION INICIAL
def mostrar_inicial(df: pd.DataFrame) -> None:
    """
    Función que muestra la información principal del DF antes de ejecutar
    las transformaciones.
    Informacion: nombre tabla, .info con info de shape, dtype y
    count valores no nulos, una muestra (sample) de 1 registros.
    """
    print(f'\n--------------- DF: PRESTAMOS HISTORICOS ---------------\n'
    "\n1. Información incial:\n"
    '\n1.1 Muestra:\n')
    display(df.sample(1))
    print("\n1.2 Información general:\n")
    df.info(memory_usage="deep")


def mostrar_normalizada(df: pd.DataFrame) -> None:
    """
    Recibe DF completo y normalizado.
    Muestra informacion post normalización: 1 registro y descripción de las
    las variables.
    """
    print(f"\n2. Resultados post normalizaciones del DF:\n"
    f"2.1 Muestra:\n")
    display(df.sample(2))


def mostrar_posibles_nulos(df: pd.DataFrame | None) -> None:
    """
    Recibe DF con el resumen nulos.
    Muestra el resumen de los posibles valores nulos encontrados.
    Si None, es que no hay posibles nulos, imprime aviso.
    """
    print(f"\n3. Primera inspección posibles nulos de la tabla:\n")
    if df is None:
      print(f"La tabla no presenta posibles valores nulos.")
    else:
      print("\nResumen posibles nulos encontrado:\n")
      display(df)


# 2. NORMALIZAR Y CAMBIAR NOMBRES COLUMNAS
def normalizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Función que normaliza los nombres de las columnas de cada df y los pasa a
    minusculas, uniendo separacion palabras con '_'.
    Imprime lista de nombres duplicados, si hay.
    """
    nuevas = []
    for c in df.columns:
      c = str(c).replace("\ufeff", "").strip().lower()
      c = unicodedata.normalize("NFKD", c)
      c = "".join(ch for ch in c if not unicodedata.combining(ch))
      c = re.sub(r"[^0-9a-z]+", "_", c).strip("_")
      nuevas.append(c)

    df = df.set_axis(nuevas, axis=1)

    if df.columns.duplicated().any():
      raise ValueError(
          f"Columnas duplicadas: {df.columns[df.columns.duplicated()].tolist()}"
          )
    del nuevas
    return df


# 3. NORMALIZAR VALORES DF
def normalizar_unicos(s: pd.Series) -> pd.Series:
    """
    Normaliza valores de texto: quita acentos/diacríticos, espacios
    sobrantes y pasa a minúsculas.
    """
    s = (s.str.normalize("NFD")
          .str.replace("[\u0300-\u036f]", "", regex=True)   # elimina diacríticos
          .str.strip()
          .str.replace(r"\s+", " ", regex=True)
          .str.lower())
    return s


def normalizar_categoria(s: pd.Series) -> pd.Series:
    """
    Normaliza los campos de tipos categoría (no las filas) y reasigna los códigos.
    Si varias categorías coinciden tras normalizar, se fusionan en una.
    """
    if len(s.cat.categories) == 0:          # columna toda nula
        return s
    cats_norm = normalizar_unicos(pd.Series(s.cat.categories).astype("string"))
    mapa, cats_unicas = pd.factorize(cats_norm)
    codigos = s.cat.codes.to_numpy()
    nuevos = np.where(codigos == -1, -1, mapa[codigos])   # -1 = nulo
    return pd.Series(
        pd.Categorical.from_codes(nuevos, categories=pd.Index(cats_unicas)),
        index=s.index, name=s.name,
    )


def normalizar_texto(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza los valores del texto, primero los valores unicos y despues
    expande al resto.
    Se eliminan espacios en blanco y se pasa todo el texto a minusculas.
    """
    for c in df.select_dtypes(["string", "category"]).columns:
      if isinstance(df[c].dtype, pd.CategoricalDtype):
        df[c] = normalizar_categoria(df[c])
      else:
        codigos, unicos = pd.factorize(df[c])
        unicos_norm = normalizar_unicos(pd.Series(unicos, dtype=df[c].dtype)).array
        df[c] = unicos_norm.take(codigos, allow_fill=True)

    return df


# 4. VER NULOS DETECCION DE NULOS IMPLICITOS / OCULTOS
def buscar_nulos(df: pd.DataFrame,
                 patrones: tuple[str, ...] = PATRONES_NULOS) -> pd.DataFrame | None:
    """
    Función que busca patrones y valores nulos (patrones
    nulos) en la tabla.
    Si no encuentra, retorna None para aviso de que no hay.
    Si encuentra, realiza resumen total valores encontrados
    por campo con posibles valores nulos y totales de cada valor.
    """
    conteos = {}
    for c in df.select_dtypes(["string", "category"]).columns:
        s = df[c]
        vc = s[s.isin(patrones)].value_counts()
        vc = vc[vc > 0]                      # category devuelve también los ceros
        if not vc.empty:
            vc.index = vc.index.astype("object")
            conteos[c] = vc

    if not conteos:
        return None

    resumen = pd.DataFrame(conteos).fillna(0).astype(int)
    resumen = resumen.reindex([p for p in patrones if p in resumen.index])
    resumen["total"] = resumen.sum(axis=1)
    resumen.loc["total"] = resumen.sum()
    resumen.index = [v if v == "total" else repr(v) for v in resumen.index]

    return resumen


# ULTITMA: FUNCION QUE ORQUESTA
def transformaciones_comunes(df: pd.DataFrame) -> tuple[
    pd.DataFrame, pd.DataFrame | None]:
    """
    Orquesta la muestra de información del DF de préstamos y sus primeras
    transformaciones y limpiezas.

    Muestra:
    - Info inicial (.info(shape, dtype, count valores no nulos), sample(1)).
    - Sample(2) del DF normalizado (nombres de campos + texto).
    - De haberlos, resumen de los posibles nulos, antes de convertirlos en
      nulos explícitos.

    Returns
    -------
    df_semilimpio : pd.DataFrame
    df_resumen_nulos : pd.DataFrame | None
        None si no se encuentran posibles nulos.
    """
    mostrar_inicial(df)
    df = normalizar_columnas(df)
    df = normalizar_texto(df)
    mostrar_normalizada(df)
    df_resumen_nulos = buscar_nulos(df)
    mostrar_posibles_nulos(df_resumen_nulos)
    df_semilimpios = df

    return df_semilimpios, df_resumen_nulos

### Quitar comillas titulos

In [ ]:
COLS_SIN_COMILLAS = (COL_TITULO, COL_AUTOR)

def quitar_comillas(df: pd.DataFrame,
                    columnas: tuple[str, ...] = COLS_SIN_COMILLAS) -> tuple[pd.DataFrame, dict[str, int]]:
    """Elimina las comillas dobles de los campos de texto libre.

    Unifica el resultado entre archivos: el motor C ya las quitaba al
    leer ('"Vértigo" /' → 'Vértigo /'), mientras que con QUOTE_NONE o
    con entrecomillado CSV correcto se conservaban.

    Returns
    -------
    df : pd.DataFrame
    cambios : dict
        columna → número de filas modificadas.
    """
    cambios = {}
    for col in columnas:
        if col not in df.columns:
            continue
        tiene = df[col].str.contains('"', regex=False).fillna(False)
        cambios[col] = int(tiene.sum())
        if cambios[col]:
            df[col] = df[col].str.replace('"', "", regex=False)
    return df, cambios

### Quitar la doble codificacion

In [ ]:
def corregir_mojibake(df: pd.DataFrame,
                      columnas: tuple[str, ...] = COLUMNAS_TEXTO
                      ) -> tuple[pd.DataFrame, dict[str, int]]:
    """Repara texto doblemente codificado ('estudiÃ³' → 'estudió')."""
    cambios = {}
    for col in columnas:
        if col not in df.columns:
            continue
        codigos, unicos = pd.factorize(df[col])
        sospechosos = pd.Series(unicos).str.contains("[ÃÂ]", regex=True)
        if not sospechosos.any():
            cambios[col] = 0
            continue
        corregidos = [ftfy.fix_encoding(u) if s else u
                      for u, s in zip(unicos, sospechosos)]
        cambios[col] = sum(a != b for a, b in zip(unicos, corregidos))
        df[col] = pd.array(corregidos, dtype="string").take(codigos,
                                                            allow_fill=True)
    return df, cambios